# Fashion-MNIST Dataset - Vision Transformer

### [Fashion-MNIST Dataset](https://github.com/zalandoresearch/fashion-mnist)

Author: [Kevin Thomas](mailto:ket189@pitt.edu)

License: MIT

## Citation

[1] Han Xiao, Kashif Rasul, Roland Vollgraf, https://github.com/zalandoresearch/fashion-mnist

[2] Alexey Dosovitskiy et al., https://arxiv.org/abs/2010.11929

## Install Libraries

In [ ]:
# conda activate prod
# conda install -c conda-forge pytorch torchvision torchaudio
# conda install numpy matplotlib scikit-learn

## Import Libraries

In [ ]:
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets
from torchvision import transforms
import matplotlib.pyplot as plt
%matplotlib inline
from sklearn.metrics import confusion_matrix
from sklearn.metrics import classification_report

## Seed

In [ ]:
SEED = 42
SEED

In [ ]:
torch.manual_seed(SEED)

## Parameters

In [ ]:
IMAGE_SIZE = 28
IMAGE_SIZE

In [ ]:
PATCH_SIZE = 4
PATCH_SIZE

In [ ]:
EMBED_DIM = 64
EMBED_DIM

In [ ]:
N_HEADS = 4
N_HEADS

In [ ]:
N_LAYERS = 4
N_LAYERS

In [ ]:
FF_DIM = 128
FF_DIM

In [ ]:
NUM_CLASSES = 10
NUM_CLASSES

In [ ]:
CLASS_NAMES = ['T-shirt/top', 'Trouser', 'Pullover', 'Dress', 'Coat',
               'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Ankle boot']
CLASS_NAMES

## Hyperparameters

In [ ]:
LEARNING_RATE = 0.001
LEARNING_RATE

In [ ]:
EPOCHS = 8
EPOCHS

In [ ]:
BATCH_SIZE = 128
BATCH_SIZE

## Device

In [ ]:
DEVICE = torch.device(
    "mps" if torch.backends.mps.is_available()
    else "cuda" if torch.cuda.is_available()
    else "cpu")
DEVICE

## Load Dataset

In [ ]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.2860,), (0.3530,))])
train_data = datasets.FashionMNIST('data', train=True, download=True, transform=transform)
test_data = datasets.FashionMNIST('data', train=False, download=True, transform=transform)
train_loader = DataLoader(train_data, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_data, batch_size=BATCH_SIZE, shuffle=False)
len(train_data), len(test_data)

## Create Patch Embedding

The image is cut into non-overlapping patches. A convolution with stride equal to the patch size turns each patch into an embedding vector.

In [ ]:
class PatchEmbedding(nn.Module):
    """
    Split an image into patches and embed each patch.
    """

    def __init__(self, embed_dim=EMBED_DIM, patch_size=PATCH_SIZE):
        """
        Initialize the patch projection.

        Parameters:
            embed_dim (int): Embedding size per patch.
            patch_size (int): Side length of each patch.

        Returns:
            None
        """
        super(PatchEmbedding, self).__init__()
        self.proj = nn.Conv2d(1, embed_dim, kernel_size=patch_size, stride=patch_size)

    def forward(self, x):
        """
        Embed a batch of images.

        Parameters:
            x (torch.Tensor): Batch of images.

        Returns:
            torch.Tensor: Patch embeddings.
        """
        x = self.proj(x)
        return x.flatten(2).transpose(1, 2)

## Create Transformer Encoder

In [ ]:
class EncoderBlock(nn.Module):
    """
    A transformer encoder block with attention and a feed-forward network.
    """

    def __init__(self, embed_dim=EMBED_DIM, n_heads=N_HEADS, ff_dim=FF_DIM):
        """
        Initialize attention, normalization, and feed-forward layers.

        Parameters:
            embed_dim (int): Embedding size.
            n_heads (int): Number of attention heads.
            ff_dim (int): Feed-forward hidden size.

        Returns:
            None
        """
        super(EncoderBlock, self).__init__()
        self.attn = nn.MultiheadAttention(embed_dim, n_heads, batch_first=True)
        self.norm1 = nn.LayerNorm(embed_dim)
        self.ff = nn.Sequential(
            nn.Linear(embed_dim, ff_dim),
            nn.GELU(),
            nn.Linear(ff_dim, embed_dim))
        self.norm2 = nn.LayerNorm(embed_dim)

    def forward(self, x):
        """
        Run the encoder block with residual connections.

        Parameters:
            x (torch.Tensor): Input of shape (batch, tokens, embed_dim).

        Returns:
            torch.Tensor: Output of the same shape.
        """
        attn_out, _ = self.attn(x, x, x)
        x = self.norm1(x + attn_out)
        return self.norm2(x + self.ff(x))

## Create Vision Transformer

In [ ]:
class ViT(nn.Module):
    """
    A small vision transformer for 28x28 grayscale images.
    """

    def __init__(self, embed_dim=EMBED_DIM, n_heads=N_HEADS, n_layers=N_LAYERS,
                 ff_dim=FF_DIM, patch_size=PATCH_SIZE, num_classes=NUM_CLASSES):
        """
        Initialize patch embedding, tokens, blocks, and the head.

        Parameters:
            embed_dim (int): Embedding size.
            n_heads (int): Number of attention heads.
            n_layers (int): Number of encoder blocks.
            ff_dim (int): Feed-forward hidden size.
            patch_size (int): Side length of each patch.
            num_classes (int): Number of output classes.

        Returns:
            None
        """
        super(ViT, self).__init__()
        self.patch = PatchEmbedding(embed_dim, patch_size)
        self.cls_token = nn.Parameter(torch.zeros(1, 1, embed_dim))
        self.pos_embed = nn.Parameter(
            torch.zeros(1, (IMAGE_SIZE // patch_size) ** 2 + 1, embed_dim))
        self.blocks = nn.Sequential(*[
            EncoderBlock(embed_dim, n_heads, ff_dim) for _ in range(n_layers)])
        self.norm = nn.LayerNorm(embed_dim)
        self.head = nn.Linear(embed_dim, num_classes)

    def forward(self, x):
        """
        Classify a batch of images.

        Parameters:
            x (torch.Tensor): Batch of images.

        Returns:
            torch.Tensor: Output logits.
        """
        x = self.patch(x)
        cls = self.cls_token.expand(x.size(0), -1, -1)
        x = torch.cat([cls, x], dim=1) + self.pos_embed
        x = self.blocks(x)
        return self.head(self.norm(x[:, 0]))

## Instantiate Model, Loss, and Optimizer

In [ ]:
torch.manual_seed(SEED)
model = ViT().to(DEVICE)
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
sum(p.numel() for p in model.parameters())

## Train Model

### Functions

In [ ]:
def train_epoch(model, loader, loss_fn, optimizer):
    """
    Train the model for a single epoch.

    Parameters:
        model (nn.Module): The model to train.
        loader (DataLoader): Training data loader.
        loss_fn (nn.Module): Loss function.
        optimizer (torch.optim.Optimizer): Parameter update rule.

    Returns:
        float: Mean training loss.
    """
    model.train()
    total = 0.0
    for x, y in loader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        optimizer.zero_grad()
        loss = loss_fn(model(x), y)
        loss.backward()
        optimizer.step()
        total += loss.item() * len(y)
    return total / len(loader.dataset)


def evaluate(model, loader, loss_fn):
    """
    Evaluate the model over a loader.

    Parameters:
        model (nn.Module): The model to evaluate.
        loader (DataLoader): Evaluation data loader.
        loss_fn (nn.Module): Loss function.

    Returns:
        tuple: Mean loss and accuracy.
    """
    model.eval()
    total, correct = 0.0, 0
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            logits = model(x)
            total += loss_fn(logits, y).item() * len(y)
            correct += (logits.argmax(1) == y).sum().item()
    n = len(loader.dataset)
    return total / n, correct / n

### Training Loop

In [ ]:
history = {'loss': [], 'accuracy': []}
for epoch in range(EPOCHS):
    train_loss = train_epoch(model, train_loader, loss_fn, optimizer)
    test_loss, test_accuracy = evaluate(model, test_loader, loss_fn)
    history['loss'].append(test_loss)
    history['accuracy'].append(test_accuracy)
    print(f"Epoch {epoch + 1} | train loss {train_loss:.4f} | test accuracy {test_accuracy:.4f}")

### Visualize Training

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(history['loss'])
axes[0].set_title('Test loss')
axes[0].set_xlabel('Epoch')
axes[1].plot(history['accuracy'])
axes[1].set_title('Test accuracy')
axes[1].set_xlabel('Epoch')
fig.tight_layout()
plt.show()

## Evaluate Model

In [ ]:
_, accuracy = evaluate(model, test_loader, loss_fn)
print(f"Test accuracy: {accuracy:.4f}")

In [ ]:
all_preds = []
all_labels = []
model.eval()
with torch.no_grad():
    for x, y in test_loader:
        x = x.to(DEVICE)
        all_preds.append(model(x).argmax(1).cpu().numpy())
        all_labels.append(y.numpy())
all_preds = np.concatenate(all_preds)
all_labels = np.concatenate(all_labels)
print(confusion_matrix(all_labels, all_preds))
print(classification_report(all_labels, all_preds, target_names=CLASS_NAMES))

## Save Model

In [ ]:
torch.save(model.state_dict(), 'vit_fashion_mnist.pt')
print('saved vit_fashion_mnist.pt')

## Load Model

In [ ]:
loaded_model = ViT().to(DEVICE)
loaded_model.load_state_dict(torch.load('vit_fashion_mnist.pt', map_location=DEVICE))
loaded_model.eval()
print('loaded vit_fashion_mnist.pt')

## Inference

### Function

In [ ]:
def predict(model, image):
    """
    Predict the class of one normalized image tensor.

    Parameters:
        model (nn.Module): Trained model.
        image (torch.Tensor): A 1x28x28 image tensor.

    Returns:
        tuple: Predicted class name and confidence.
    """
    model.eval()
    with torch.no_grad():
        logits = model(image.unsqueeze(0).to(DEVICE))
        probs = torch.softmax(logits, dim=1)
        confidence, predicted = probs.max(dim=1)
    return CLASS_NAMES[predicted.item()], confidence.item()

### Run Inference

In [ ]:
image, label = test_data[12]
name, confidence = predict(loaded_model, image)
print(f"True: {CLASS_NAMES[label]} | Predicted: {name} ({confidence:.4f})")
plt.imshow(image.squeeze(), cmap='gray')
plt.title(f"Predicted: {name}")
plt.axis('off')
plt.show()